<a href="https://colab.research.google.com/github/aparna-2001/GATE_PPI/blob/main/graph_construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
import pandas as pd

base_dir = '/content/drive/MyDrive/ppi_gnn'

# 1. The full-backbone structure data — what you just fetched (the main one you need)
with open(f'{base_dir}/data/raw/full_backbone_checkpoint.json', 'r') as f:
    full_structure_data = json.load(f)
print(f"full_structure_data loaded: {len(full_structure_data):,} proteins")

# 2. The original structure_data (source/status/pLDDT metadata per protein)
#    — useful for cross-checking source (PDB/AF2/ESMFold) and quality notes
with open(f'{base_dir}/data/raw/structure_data_final.json', 'r') as f:
    structure_data = json.load(f)
print(f"structure_data loaded: {len(structure_data):,} proteins")

# 3. The final split-assigned pairs — what graphs actually need to support
pairs_split_final = pd.read_parquet(f'{base_dir}/data/labels/pairs_split_final.parquet')
print(f"pairs_split_final loaded: {len(pairs_split_final):,} pairs")

# 4. The protein -> split mapping (train/val/test), needed to tag each graph
with open(f'{base_dir}/data/labels/protein_split_mapping.json', 'r') as f:
    protein_split = json.load(f)
print(f"protein_split loaded: {len(protein_split):,} proteins")

Mounted at /content/drive
full_structure_data loaded: 2,608 proteins
structure_data loaded: 4,403 proteins
pairs_split_final loaded: 6,808 pairs
protein_split loaded: 2,608 proteins


In [ ]:
unique_proteins_in_pairs = set(pairs_split_final['protein1']) | set(pairs_split_final['protein2'])
print(f"Unique proteins actually needed for graphs: {len(unique_proteins_in_pairs):,}")

missing = unique_proteins_in_pairs - set(full_structure_data.keys())
print(f"Proteins needed but missing from full_structure_data: {len(missing):,}")
if missing:
    print(f"Sample missing: {list(missing)[:5]}")

Unique proteins actually needed for graphs: 2,227
Proteins needed but missing from full_structure_data: 0


In [ ]:
import numpy as np

AA_ORDER = ['ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
            'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL']
AA_TO_IDX = {aa: i for i, aa in enumerate(AA_ORDER)}

def aa_one_hot(resname):
    vec = np.zeros(21)  # 20 standard + 1 unknown
    idx = AA_TO_IDX.get(resname, 20)
    vec[idx] = 1.0
    return vec

def compute_dihedral(p0, p1, p2, p3):
    """Standard dihedral angle calculation from 4 points (in degrees)."""
    p0, p1, p2, p3 = [np.array(p) for p in (p0, p1, p2, p3)]
    b0 = p0 - p1
    b1 = p2 - p1
    b2 = p3 - p2
    b1 /= np.linalg.norm(b1)
    v = b0 - np.dot(b0, b1) * b1
    w = b2 - np.dot(b2, b1) * b1
    x = np.dot(v, w)
    y = np.dot(np.cross(b1, v), w)
    return np.degrees(np.arctan2(y, x))

def compute_phi_psi(residues):
    """
    Computes phi/psi for every residue in a chain.
    phi needs C(i-1), N(i), CA(i), C(i)
    psi needs N(i), CA(i), C(i), N(i+1)
    Terminal residues get NaN where the neighbor is missing.
    """
    n = len(residues)
    phi = [np.nan] * n
    psi = [np.nan] * n

    for i in range(n):
        # phi: needs previous residue's C
        if i > 0 and residues[i-1]['C'] is not None and residues[i]['N'] is not None and residues[i]['CA'] is not None and residues[i]['C'] is not None:
            phi[i] = compute_dihedral(residues[i-1]['C'], residues[i]['N'], residues[i]['CA'], residues[i]['C'])
        # psi: needs next residue's N
        if i < n-1 and residues[i]['N'] is not None and residues[i]['CA'] is not None and residues[i]['C'] is not None and residues[i+1]['N'] is not None:
            psi[i] = compute_dihedral(residues[i]['N'], residues[i]['CA'], residues[i]['C'], residues[i+1]['N'])

    return phi, psi

def dihedral_features(phi, psi):
    """Encode as [sin(phi), cos(phi), sin(psi), cos(psi)], 0 where missing."""
    n = len(phi)
    feat = np.zeros((n, 4))
    for i in range(n):
        if not np.isnan(phi[i]):
            feat[i, 0] = np.sin(np.radians(phi[i]))
            feat[i, 1] = np.cos(np.radians(phi[i]))
        if not np.isnan(psi[i]):
            feat[i, 2] = np.sin(np.radians(psi[i]))
            feat[i, 3] = np.cos(np.radians(psi[i]))
    return feat

print("Feature functions defined")

Feature functions defined


In [ ]:
test_uid = list(full_structure_data.keys())[0]
test_residues = full_structure_data[test_uid]['residues']

phi, psi = compute_phi_psi(test_residues)
dihedral_feat = dihedral_features(phi, psi)

print(f"Testing on: {test_uid} ({len(test_residues)} residues)")
print(f"Dihedral feature shape: {dihedral_feat.shape}")
print(f"First residue (should be ~0, no phi): {dihedral_feat[0]}")
print(f"Middle residue: {dihedral_feat[len(test_residues)//2]}")
print(f"phi range: {np.nanmin(phi):.1f} to {np.nanmax(phi):.1f}")
print(f"psi range: {np.nanmin(psi):.1f} to {np.nanmax(psi):.1f}")

Testing on: Q8NCQ5 (510 residues)
Dihedral feature shape: (510, 4)
First residue (should be ~0, no phi): [ 0.          0.          0.93995668 -0.34129377]
Middle residue: [-0.92612665 -0.3772127   0.66239015 -0.74915906]
phi range: -174.0 to 175.8
psi range: -176.4 to 177.3


In [ ]:
def build_node_features(uid, full_structure_data, structure_data):
    """
    Builds the full node feature matrix for one protein.
    Returns: (N, 27) array -- 21 AA one-hot + 4 dihedral + 1 pLDDT + 2 source flag... wait let's check the actual dims
    """
    residues = full_structure_data[uid]['residues']
    n = len(residues)

    # 1. AA one-hot (21-dim)
    aa_feat = np.array([aa_one_hot(r['resname']) for r in residues])

    # 2. Dihedrals (4-dim)
    phi, psi = compute_phi_psi(residues)
    dihedral_feat = dihedral_features(phi, psi)

    # 3. pLDDT (1-dim) -- normalize to 0-1
    source = structure_data[uid]['source']
    if source == 'PDB':
        plddt_feat = np.ones((n, 1))  # PDB = fully confident
    else:
        raw_bfactors = np.array([r['bfactor'] for r in residues])
        # ESMFold stores 0-1, AF2 stores 0-100 -- normalize both to 0-1
        if raw_bfactors.max() <= 1.0:
            plddt_feat = raw_bfactors.reshape(-1, 1)
        else:
            plddt_feat = (raw_bfactors / 100.0).reshape(-1, 1)

    # 4. Source flag (2-dim): [is_PDB, is_predicted(AF2 or ESMFold)]
    if source == 'PDB':
        source_feat = np.tile([1.0, 0.0], (n, 1))
    else:
        source_feat = np.tile([0.0, 1.0], (n, 1))

    node_features = np.concatenate([aa_feat, dihedral_feat, plddt_feat, source_feat], axis=1)
    return node_features


# Test on the same protein
test_features = build_node_features('Q8NCQ5', full_structure_data, structure_data)
print(f"Node feature matrix shape: {test_features.shape}")
print(f"Expected: (510, 28)  [21 AA + 4 dihedral + 1 pLDDT + 2 source]")
print(f"\nFirst residue features:\n{test_features[0]}")
print(f"\npLDDT column stats: min={test_features[:,25].min():.3f}, max={test_features[:,25].max():.3f}, mean={test_features[:,25].mean():.3f}")

Node feature matrix shape: (510, 28)
Expected: (510, 28)  [21 AA + 4 dihedral + 1 pLDDT + 2 source]

First residue features:
[ 0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  1.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.93995668
 -0.34129377  0.3173      0.          1.        ]

pLDDT column stats: min=0.194, max=0.971, mean=0.766


In [ ]:
import time

node_features_all = {}
node_feature_errors = []

needed_proteins = list(unique_proteins_in_pairs)
print(f"Building node features for {len(needed_proteins):,} proteins...")

start = time.time()
for i, uid in enumerate(needed_proteins):
    try:
        feat = build_node_features(uid, full_structure_data, structure_data)
        node_features_all[uid] = feat
    except Exception as e:
        node_feature_errors.append((uid, str(e)))

    if (i+1) % 500 == 0:
        elapsed = time.time() - start
        print(f"  Processed {i+1}/{len(needed_proteins)} ({elapsed:.1f}s elapsed)")

print(f"\nDone. Successful: {len(node_features_all):,}")
print(f"Errors: {len(node_feature_errors):,}")
if node_feature_errors:
    print(f"Sample errors: {node_feature_errors[:5]}")

Building node features for 2,227 proteins...
  Processed 500/2227 (23.9s elapsed)
  Processed 1000/2227 (48.7s elapsed)
  Processed 1500/2227 (70.8s elapsed)
  Processed 2000/2227 (94.4s elapsed)

Done. Successful: 2,226
Errors: 1
Sample errors: [('O75822', "object of type 'NoneType' has no len()")]


In [ ]:
print(structure_data.get('O75822'))
print()
print(full_structure_data.get('O75822'))

{'source': 'PDB', 'status': 'success', 'n_residues': 160, 'quality_note': None}

{'source': 'PDB', 'residues': None, 'n_residues': 0}


In [ ]:
if 'O75822' in full_structure_data:
    print(f"residues is None: {full_structure_data['O75822']['residues'] is None}")
    print(f"source: {full_structure_data['O75822']['source']}")

residues is None: True
source: PDB


In [ ]:
pairs_affected = pairs_split_final[
    (pairs_split_final['protein1'] == 'O75822') | (pairs_split_final['protein2'] == 'O75822')
]
print(f"Pairs involving O75822: {len(pairs_affected)}")
print(pairs_affected[['protein1', 'protein2', 'label', 'split']])

Pairs involving O75822: 13
     protein1    protein2  label  split
1630   Q9HA81      O75822      0  train
1881   O75822      Q9NWA0      0  train
1885   O75822      Q96GT1      0  train
2105   O75822      P16035      0  train
3231   P02649      O75822      0  train
3413   O75822      O15446      0  train
3543   O75822      O00703      0  train
3626   O75822      P23528      0  train
3859   O75822      P25490      0  train
5223   O75822      P0DP04      0  train
5454   P51857      O75822      0  train
5943   Q8IX34      O75822      0  train
6060   O75822  A0A1W2PRB8      0  train


In [ ]:
# Drop O75822 from the node feature pool and from pairs
pairs_split_final_clean = pairs_split_final[
    (pairs_split_final['protein1'] != 'O75822') & (pairs_split_final['protein2'] != 'O75822')
].copy()

print(f"Pairs before dropping O75822: {len(pairs_split_final):,}")
print(f"Pairs after dropping O75822:  {len(pairs_split_final_clean):,}")
print(f"Dropped: {len(pairs_split_final) - len(pairs_split_final_clean):,}")

print(f"\nSplit distribution after drop:")
print(pairs_split_final_clean['split'].value_counts())

Pairs before dropping O75822: 6,808
Pairs after dropping O75822:  6,795
Dropped: 13

Split distribution after drop:
split
train    6563
test      119
val       113
Name: count, dtype: int64


In [ ]:
# Update unique_proteins_in_pairs to match the cleaned pairs
unique_proteins_in_pairs = set(pairs_split_final_clean['protein1']) | set(pairs_split_final_clean['protein2'])
print(f"\nUnique proteins needed now: {len(unique_proteins_in_pairs):,}")
print(f"O75822 still needed? {'O75822' in unique_proteins_in_pairs}")


Unique proteins needed now: 2,226
O75822 still needed? False


In [ ]:
import pickle
import os

output_dir = f'{base_dir}/data/processed/'
os.makedirs(output_dir, exist_ok=True)

output_filepath = f'{output_dir}/node_features_step8b.pkl'
with open(output_filepath, 'wb') as f:
    pickle.dump(node_features_all, f)

print(f"Saved node features for {len(node_features_all):,} proteins")
print(f"File size: {os.path.getsize(output_filepath) / 1e6:.1f} MB")

Saved node features for 2,226 proteins
File size: 156.0 MB


In [ ]:
!apt-get install -y dssp 2>&1 | tail -5

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_opencl.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link

Processing triggers for man-db (2.10.2-1) ...


In [ ]:
!which mkdssp
!which dssp
!dpkg -L dssp 2>/dev/null | grep bin

/usr/bin/mkdssp
/usr/bin/dssp
/usr/bin
/usr/bin/mkdssp
/usr/bin/dssp


In [ ]:
!pip install biopython

import os
from Bio.PDB import PDBParser
from Bio.PDB.DSSP import DSSP

def write_minimal_pdb(uid, residues, out_path):
    """
    Writes a minimal PDB file containing just N, CA, C, O atoms,
    enough for DSSP to compute secondary structure and approximate SASA.
    """
    lines = []
    atom_num = 1
    for i, r in enumerate(residues):
        resseq = r['resseq']
        resname = r['resname']
        for atom_name, coord in [('N', r['N']), ('CA', r['CA']), ('C', r['C']), ('O', r['O'])]:
            if coord is None:
                continue
            x, y, z = coord
            lines.append(
                f"ATOM  {atom_num:5d}  {atom_name:<3s}{resname:>3s} A{resseq:4d}    "
                f"{x:8.3f}{y:8.3f}{z:8.3f}  1.00  0.00           {atom_name[0]}"
            )
            atom_num += 1
    lines.append("TER")
    lines.append("END")

    with open(out_path, 'w') as f:
        f.write('\n'.join(lines) + '\n')


def run_dssp_on_protein(uid, residues):
    """Writes temp PDB, runs DSSP, returns per-residue SS and SASA, then cleans up."""
    tmp_pdb = f'/content/tmp_dssp_{uid}.pdb'
    write_minimal_pdb(uid, residues, tmp_pdb)

    try:
        parser = PDBParser(QUIET=True)
        structure = parser.get_structure(uid, tmp_pdb)
        model = structure[0]
        dssp = DSSP(model, tmp_pdb)

        # DSSP keys are (chain_id, residue_id); build an ordered list matching our residues
        ss_list = []
        sasa_list = []
        dssp_keys = list(dssp.keys())
        key_lookup = {k[1][1]: k for k in dssp_keys}  # resseq -> full key

        for r in residues:
            key = key_lookup.get(r['resseq'])
            if key is not None:
                data = dssp[key]
                ss_list.append(data[2])      # secondary structure code
                sasa_list.append(data[3])    # relative SASA
            else:
                ss_list.append('-')
                sasa_list.append(0.0)

        return ss_list, sasa_list
    except Exception as e:
        return None, str(e)
    finally:
        if os.path.exists(tmp_pdb):
            os.remove(tmp_pdb)


# Test on one protein
test_uid = 'Q8NCQ5'
test_residues = full_structure_data[test_uid]['residues']
ss_list, sasa_list = run_dssp_on_protein(test_uid, test_residues)

if ss_list is None:
    print(f"DSSP failed: {sasa_list}")  # error message stored here on failure
else:
    print(f"DSSP succeeded on {test_uid}")
    print(f"SS codes (first 20): {ss_list[:20]}")
    print(f"SASA (first 10): {[round(s, 3) for s in sasa_list[:10]]}")
    from collections import Counter
    print(f"\nSS code distribution: {Counter(ss_list)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 68.9 MB/s eta 0:00:00
DSSP failed: DSSP failed to produce an output


/usr/local/lib/python3.12/dist-packages/Bio/PDB/DSSP.py:199: UserWarning: Line number 2042 is empty!
Dropped unsupported records: TER
Could not assign value ' ' to column _pdbx_nonpoly_scheme.pdb_strand_id
Error parsing PDB at line 1
Error trying to load file "/content/tmp_dssp_Q8NCQ5.pdb"
When validating _pdbx_nonpoly_scheme.pdb_strand_id: Value ' ' does not match type expression for type code

  warnings.warn(err)


In [ ]:
from Bio.PDB.DSSP import DSSP
help(DSSP.__init__)

Help on function __init__ in module Bio.PDB.DSSP:

__init__(self, model, in_file, dssp='dssp', acc_array='Sander', file_type='')
    Create a DSSP object.

    Parameters
    ----------
    model : Model
        The first model of the structure
    in_file : string
        Either a PDB file or a DSSP file.
    dssp : string
        The dssp executable (ie. the argument to subprocess)
    acc_array : string
        Accessible surface area (ASA) from either Miller et al. (1987),
        Sander & Rost (1994), Wilke: Tien et al. 2013, or Ahmad et al.
        (2003) as string Sander/Wilke/Miller/Ahmad. Defaults to Sander.
    file_type: string
        File type switch: either PDB, MMCIF or DSSP. Inferred from the
        file extension by default.



In [ ]:
def run_dssp_on_protein(uid, residues):
    """Writes temp PDB, runs DSSP, returns per-residue SS and SASA, then cleans up."""
    tmp_pdb = f'/content/tmp_dssp_{uid}.pdb'
    write_minimal_pdb(uid, residues, tmp_pdb)

    try:
        parser = PDBParser(QUIET=True)
        structure = parser.get_structure(uid, tmp_pdb)
        model = structure[0]
        dssp = DSSP(model, tmp_pdb, dssp='mkdssp')

        ss_list = []
        sasa_list = []
        dssp_keys = list(dssp.keys())
        key_lookup = {k[1][1]: k for k in dssp_keys}

        for r in residues:
            key = key_lookup.get(r['resseq'])
            if key is not None:
                data = dssp[key]
                ss_list.append(data[2])
                sasa_list.append(data[3])
            else:
                ss_list.append('-')
                sasa_list.append(0.0)

        return ss_list, sasa_list
    except Exception as e:
        return None, str(e)
    finally:
        if os.path.exists(tmp_pdb):
            os.remove(tmp_pdb)


# Re-test on the same protein
test_uid = 'Q8NCQ5'
test_residues = full_structure_data[test_uid]['residues']
ss_list, sasa_list = run_dssp_on_protein(test_uid, test_residues)

if ss_list is None:
    print(f"DSSP failed: {sasa_list}")
else:
    print(f"DSSP succeeded on {test_uid}")
    print(f"SS codes (first 20): {ss_list[:20]}")
    print(f"SASA (first 10): {[round(s, 3) for s in sasa_list[:10]]}")
    from collections import Counter
    print(f"\nSS code distribution: {Counter(ss_list)}")

DSSP failed: DSSP failed to produce an output


/usr/local/lib/python3.12/dist-packages/Bio/PDB/DSSP.py:199: UserWarning: Line number 2042 is empty!
Dropped unsupported records: TER
Could not assign value ' ' to column _pdbx_nonpoly_scheme.pdb_strand_id
Error parsing PDB at line 1
Error trying to load file "/content/tmp_dssp_Q8NCQ5.pdb"
When validating _pdbx_nonpoly_scheme.pdb_strand_id: Value ' ' does not match type expression for type code

  warnings.warn(err)


In [ ]:
!mkdssp --version

mkdssp version 4.0.4


In [ ]:
!mkdssp --help 2>&1 | head -30

mkdssp [options] input-file [output-file]:
  --dict arg            Dictionary file containing restraints for residues in 
                        this specific target, can be specified multiple times.
  --output-format arg   Output format, can be either 'dssp' for classic DSSP or
                        'mmcif' for annotated mmCIF. The default is chosen 
                        based on the extension of the output file, if any.
  --min-pp-stretch arg  Minimal number of residues having PSI/PHI in range for 
                        a PP helix, default is 3
  --write-other         If set, write the type OTHER for loops, default is to 
                        leave this out
  -h [ --help ]         Display help message
  --version             Print version
  -v [ --verbose ]      verbose output



In [ ]:
from Bio.PDB import StructureBuilder
from Bio.PDB.PDBIO import PDBIO
from Bio.PDB.Atom import Atom
import warnings
import os
warnings.filterwarnings('ignore')

CRYST1_LINE = "CRYST1    1.000    1.000    1.000  90.00  90.00  90.00 P 1           1\n"

def build_biopython_structure(uid, residues):
    """Builds an in-memory BioPython Structure from our stored coordinates,
    so we can write a properly-formatted PDB file via PDBIO (avoids hand-rolled
    string formatting issues that newer mkdssp versions are strict about)."""
    sb = StructureBuilder.StructureBuilder()
    sb.init_structure(uid)
    sb.init_model(0)
    sb.init_chain('A')
    sb.init_seg(' ')

    # Use a dictionary to store residues, keyed by resseq, to handle potential duplicates
    # and ensure only one residue per resseq is added. If multiple residues have the same
    # resseq, the last one encountered will overwrite previous ones.
    unique_residues_by_resseq = {}
    for r in residues:
        resseq = r['resseq']
        unique_residues_by_resseq[resseq] = r

    # Sort residues by resseq before adding them to maintain sequence order
    sorted_residues = sorted(unique_residues_by_resseq.values(), key=lambda x: x['resseq'])

    for r in sorted_residues:
        resname = r['resname']
        resseq = r['resseq']
        icode = ' ' # Assuming no insertion codes

        # BioPython init_residue takes (resname, icode, resseq, segment_id)
        # We ensure resseq is unique in the loop by using unique_residues_by_resseq.
        sb.init_residue(resname, icode, resseq, ' ') # segment_id

        for atom_name, coord in [('N', r['N']), ('CA', r['CA']), ('C', r['C']), ('O', r['O'])]:
            if coord is None:
                continue
            # For atoms, altloc is typically ' ' if not specified
            sb.init_atom(atom_name, coord, 0.0, 1.0, ' ', f' {atom_name:<3s}', element=atom_name[0])

    return sb.get_structure()


def write_minimal_pdb_v2(uid, residues, out_path):
    structure = build_biopython_structure(uid, residues)
    io = PDBIO()
    io.set_structure(structure)
    io.save(out_path)

    # Prepend CRYST1 record, required by mkdssp 4.0.4+
    with open(out_path, 'r') as f:
        content = f.read()
    with open(out_path, 'w') as f:
        f.write(CRYST1_LINE + content)


def run_dssp_on_protein_v2(uid, residues):
    tmp_pdb = f'/content/tmp_dssp_{uid}.pdb'
    write_minimal_pdb_v2(uid, residues, tmp_pdb)

    try:
        parser = PDBParser(QUIET=True)
        structure = parser.get_structure(uid, tmp_pdb)
        model = structure[0]
        dssp = DSSP(model, tmp_pdb, dssp='mkdssp')

        ss_list = []
        sasa_list = []
        dssp_keys = list(dssp.keys())
        key_lookup = {k[1][1]: k for k in dssp_keys}

        for r in residues:
            key = key_lookup.get(r['resseq'])
            if key is not None:
                data = dssp[key]
                ss_list.append(data[2])
                sasa_list.append(data[3])
            else:
                ss_list.append('-')
                sasa_list.append(0.0)

        return ss_list, sasa_list
    except Exception as e:
        return None, str(e)
    finally:
        if os.path.exists(tmp_pdb):
            os.remove(tmp_pdb)


# Re-test
test_uid = 'Q8NCQ5'
test_residues = full_structure_data[test_uid]['residues']
ss_list, sasa_list = run_dssp_on_protein_v2(test_uid, test_residues)

if ss_list is None:
    print(f"DSSP failed: {sasa_list}")
else:
    print(f"DSSP succeeded on {test_uid}")
    print(f"SS codes (first 20): {ss_list[:20]}")
    print(f"SASA (first 10): {[round(s, 3) for s in sasa_list[:10]]}")
    from collections import Counter
    print(f"\nSS code distribution: {Counter(ss_list)}")

DSSP succeeded on Q8NCQ5
SS codes (first 20): ['-', '-', '-', '-', 'S', 'S', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'T', 'T', '-']
SASA (first 10): [0.755, 0.755, 0.655, 0.845, 0.347, 0.833, 0.242, 0.42, 0.402, 0.273]

SS code distribution: Counter({'E': 160, '-': 113, 'H': 104, 'T': 54, 'S': 43, 'G': 19, 'P': 14, 'B': 3})


In [ ]:
test_uid = 'Q8NCQ5'
test_residues = full_structure_data[test_uid]['residues']
tmp_pdb = f'/content/tmp_dssp_{test_uid}.pdb'
write_minimal_pdb_v2(test_uid, test_residues, tmp_pdb)

# Peek at the actual file content first
with open(tmp_pdb) as f:
    lines = f.readlines()
print(f"Total lines: {len(lines)}")
print(''.join(lines[:10]))
print("...")
print(''.join(lines[-5:]))

Total lines: 2043
CRYST1    1.000    1.000    1.000  90.00  90.00  90.00 P 1           1
ATOM      1  N   MET A   1      61.490 -17.505 -13.657  1.00  0.00           N  
ATOM      2  CA  MET A   1      62.366 -16.335 -13.427  1.00  0.00           C  
ATOM      3  C   MET A   1      62.075 -15.279 -14.483  1.00  0.00           C  
ATOM      4  O   MET A   1      62.358 -15.544 -15.637  1.00  0.00           O  
ATOM      5  N   ALA A   2      61.463 -14.152 -14.104  1.00  0.00           N  
ATOM      6  CA  ALA A   2      61.555 -12.838 -14.766  1.00  0.00           C  
ATOM      7  C   ALA A   2      60.602 -11.867 -14.044  1.00  0.00           C  
ATOM      8  O   ALA A   2      59.472 -11.604 -14.450  1.00  0.00           O  
ATOM      9  N   THR A   3      61.068 -11.374 -12.904  1.00  0.00           N  

...
ATOM   2038  CA  TYR A 510      -8.849 -16.835  22.556  1.00  0.00           C  
ATOM   2039  C   TYR A 510      -9.562 -17.773  21.593  1.00  0.00           C  
ATOM   2040  O 

In [ ]:
# Run mkdssp directly to see the real error
!mkdssp /content/tmp_dssp_Q8NCQ5.pdb /content/tmp_dssp_Q8NCQ5.dssp

In [ ]:
CRYST1_LINE = "CRYST1    1.000    1.000    1.000  90.00  90.00  90.00 P 1           1\n"

def write_minimal_pdb_v3(uid, residues, out_path):
    structure = build_biopython_structure(uid, residues)
    io = PDBIO()
    io.set_structure(structure)
    io.save(out_path)

    # Prepend CRYST1 record, required by mkdssp 4.0.4+
    with open(out_path, 'r') as f:
        content = f.read()
    with open(out_path, 'w') as f:
        f.write(CRYST1_LINE + content)


def run_dssp_on_protein_v3(uid, residues):
    tmp_pdb = f'/content/tmp_dssp_{uid}.pdb'
    write_minimal_pdb_v3(uid, residues, tmp_pdb)

    try:
        parser = PDBParser(QUIET=True)
        structure = parser.get_structure(uid, tmp_pdb)
        model = structure[0]
        dssp = DSSP(model, tmp_pdb, dssp='mkdssp')

        ss_list = []
        sasa_list = []
        dssp_keys = list(dssp.keys())
        key_lookup = {k[1][1]: k for k in dssp_keys}

        for r in residues:
            key = key_lookup.get(r['resseq'])
            if key is not None:
                data = dssp[key]
                ss_list.append(data[2])
                sasa_list.append(data[3])
            else:
                ss_list.append('-')
                sasa_list.append(0.0)

        return ss_list, sasa_list
    except Exception as e:
        return None, str(e)
    finally:
        if os.path.exists(tmp_pdb):
            os.remove(tmp_pdb)


# Re-test
test_uid = 'Q8NCQ5'
test_residues = full_structure_data[test_uid]['residues']
ss_list, sasa_list = run_dssp_on_protein_v3(test_uid, test_residues)

if ss_list is None:
    print(f"DSSP failed: {sasa_list}")
else:
    print(f"DSSP succeeded on {test_uid}")
    print(f"SS codes (first 20): {ss_list[:20]}")
    print(f"SASA (first 10): {[round(s, 3) for s in sasa_list[:10]]}")
    from collections import Counter
    print(f"\nSS code distribution: {Counter(ss_list)}")

DSSP succeeded on Q8NCQ5
SS codes (first 20): ['-', '-', '-', '-', 'S', 'S', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'T', 'T', '-']
SASA (first 10): [0.755, 0.755, 0.655, 0.845, 0.347, 0.833, 0.242, 0.42, 0.402, 0.273]

SS code distribution: Counter({'E': 160, '-': 113, 'H': 104, 'T': 54, 'S': 43, 'G': 19, 'P': 14, 'B': 3})


In [ ]:
import time

dssp_results = {}
dssp_errors = []

needed_proteins_list = list(unique_proteins_in_pairs)
print(f"Running DSSP on {len(needed_proteins_list):,} proteins...")

start = time.time()
for i, uid in enumerate(needed_proteins_list):
    residues = full_structure_data[uid]['residues']
    ss_list, sasa_list = run_dssp_on_protein_v3(uid, residues)

    if ss_list is None:
        dssp_errors.append((uid, sasa_list))  # error message stored here
    else:
        dssp_results[uid] = {'ss': ss_list, 'sasa': sasa_list}

    if (i+1) % 200 == 0:
        elapsed = time.time() - start
        print(f"  Processed {i+1}/{len(needed_proteins_list)} ({elapsed:.1f}s elapsed, {len(dssp_errors)} errors so far)")

print(f"\nDone. Successful: {len(dssp_results):,}")
print(f"Errors: {len(dssp_errors):,}")
if dssp_errors:
    print(f"Sample errors: {dssp_errors[:5]}")

Running DSSP on 2,226 proteins...
  Processed 200/2226 (171.0s elapsed, 0 errors so far)
  Processed 400/2226 (336.7s elapsed, 0 errors so far)
  Processed 600/2226 (499.7s elapsed, 0 errors so far)


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



KeyboardInterrupt: 

In [ ]:
import pickle
import os

with open(f'{base_dir}/data/processed/dssp_results_step8c.pkl', 'wb') as f:
    pickle.dump(dssp_results, f)

print(f"Saved DSSP results for {len(dssp_results):,} proteins")
print(f"File size: {os.path.getsize(f'{base_dir}/data/processed/dssp_results_step8c.pkl') / 1e6:.1f} MB")

In [ ]:
SS_CLASSES = ['H', 'B', 'E', 'G', 'I', 'T', 'S', '-']
SS_TO_IDX = {c: i for i, c in enumerate(SS_CLASSES)}

def ss_one_hot(ss_code):
    vec = np.zeros(8)
    # Fold DSSP 4.x's 'P' (polyproline helix) into coil/'-', and any other
    # unrecognized code, since it's outside the classic 8-class scheme
    idx = SS_TO_IDX.get(ss_code, SS_TO_IDX['-'])
    vec[idx] = 1.0
    return vec

def build_dssp_features(uid, dssp_results):
    result = dssp_results[uid]
    n = len(result['ss'])
    ss_feat = np.array([ss_one_hot(s) for s in result['ss']])
    sasa_feat = np.array(result['sasa']).reshape(-1, 1)
    return np.concatenate([ss_feat, sasa_feat], axis=1)  # (n, 9)


# Test and check P-code fold worked
# Use a test_uid that is guaranteed to be in dssp_results
test_uid = list(dssp_results.keys())[0]
dssp_feat = build_dssp_features(test_uid, dssp_results)
print(f"DSSP feature shape: {dssp_feat.shape}  (expected ({len(dssp_results[test_uid]['ss'])}, 9))")
print(f"Sum check (each row should sum to ~1.0 + sasa): {dssp_feat[0].sum():.3f}")

# Confirm P-coded residues correctly folded into the '-' column (index 7)
p_indices = [i for i, s in enumerate(dssp_results[test_uid]['ss']) if s == 'P']
print(f"\nResidues with raw 'P' code: {len(p_indices)}")
if p_indices:
    print(f"Their one-hot vectors (SS part only): {dssp_feat[p_indices[0], :8]}")

In [ ]:
import pickle
import os

output_dir = f'{base_dir}/data/processed/'
os.makedirs(output_dir, exist_ok=True)

output_filepath = f'{output_dir}/dssp_features_step8c.pkl'
with open(output_filepath, 'wb') as f:
    pickle.dump(dssp_results, f)

print(f"Saved DSSP features for {len(dssp_results):,} proteins")
print(f"File size: {os.path.getsize(output_filepath) / 1e6:.1f} MB")

In [ ]:
# Pick a real protein from the set we just processed
test_uid = needed_proteins_list[0]
print(f"Using: {test_uid}")
print(f"Residue count in full_structure_data: {len(full_structure_data[test_uid]['residues'])}")
print(f"Residue count in dssp_results:        {len(dssp_results[test_uid]['ss'])}")

dssp_feat = build_dssp_features(test_uid, dssp_results)
print(f"\nDSSP feature shape: {dssp_feat.shape}")
print(f"Sum check (SS one-hot + SASA, expect ~1.0 to ~2.0): {dssp_feat[0].sum():.3f}")

p_indices = [i for i, s in enumerate(dssp_results[test_uid]['ss']) if s == 'P']
print(f"\nResidues with raw 'P' code: {len(p_indices)}")
if p_indices:
    print(f"Their one-hot vectors (SS part only): {dssp_feat[p_indices[0], :8]}")

In [ ]:
dssp_features_all = {}
dssp_feature_errors = []

for uid in needed_proteins_list:
    try:
        dssp_features_all[uid] = build_dssp_features(uid, dssp_results)
    except Exception as e:
        dssp_feature_errors.append((uid, str(e)))

print(f"DSSP features built: {len(dssp_features_all):,}")
print(f"Errors: {len(dssp_feature_errors):,}")

In [ ]:
# Merge Step 8B (28-dim) + Step 8C (9-dim) into a combined (N, 37) matrix per protein
combined_features = {}
merge_errors = []

for uid in needed_proteins_list:
    try:
        feat_8b = node_features_all[uid]   # (n, 28)
        feat_8c = dssp_features_all[uid]   # (n, 9)

        if feat_8b.shape[0] != feat_8c.shape[0]:
            merge_errors.append((uid, f"shape mismatch: {feat_8b.shape[0]} vs {feat_8c.shape[0]}"))
            continue

        combined_features[uid] = np.concatenate([feat_8b, feat_8c], axis=1)  # (n, 37)
    except KeyError as e:
        merge_errors.append((uid, f"missing key: {e}"))

print(f"Combined features built: {len(combined_features):,}")
print(f"Merge errors: {len(merge_errors):,}")
if merge_errors:
    print(f"Sample errors: {merge_errors[:5]}")

# Confirm final shape on our verified test protein
print(f"\n{test_uid} combined shape: {combined_features[test_uid].shape}  (expect (725, 37))")

In [ ]:
!pip install fair-esm --quiet

In [ ]:
import torch
import esm

print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

model, alphabet = esm.pretrained.esm2_t12_35M_UR50D()
model.eval()
if torch.cuda.is_available():
    model = model.cuda()

batch_converter = alphabet.get_batch_converter()
print("ESM-2 t12 (35M, 480-dim) loaded successfully")

In [ ]:
import requests
import time

protein_sequences_esm = {}
seq_fetch_errors = []

print(f"Fetching sequences for {len(needed_proteins_list):,} proteins...")

for i, uid in enumerate(needed_proteins_list):
    try:
        resp = requests.get(f"https://rest.uniprot.org/uniprotkb/{uid}.json", timeout=10)
        if resp.status_code == 200:
            seq = resp.json().get('sequence', {}).get('value', None)
            if seq:
                protein_sequences_esm[uid] = seq
            else:
                seq_fetch_errors.append(uid)
        else:
            seq_fetch_errors.append(uid)
    except Exception:
        seq_fetch_errors.append(uid)

    if (i+1) % 300 == 0:
        print(f"  Fetched {i+1}/{len(needed_proteins_list)}...")
    time.sleep(0.06)

print(f"\nSequences retrieved: {len(protein_sequences_esm):,}")
print(f"Errors: {len(seq_fetch_errors):,}")

In [ ]:
length_mismatches = []
for uid in protein_sequences_esm:
    seq_len = len(protein_sequences_esm[uid])
    struct_len = len(full_structure_data[uid]['residues'])
    if seq_len != struct_len:
        length_mismatches.append((uid, seq_len, struct_len))

print(f"Length mismatches: {len(length_mismatches):,}")
if length_mismatches:
    print(f"Sample: {length_mismatches[:5]}")

In [ ]:
# Check a few mismatches more closely -- is the structure shorter because of
# missing residues (gaps), or because of a completely different reason
# (e.g., multi-domain construct, wrong chain selected, etc.)

for uid, seq_len, struct_len in length_mismatches[:5]:
    residues = full_structure_data[uid]['residues']
    resseq_values = [r['resseq'] for r in residues]
    print(f"{uid}: seq_len={seq_len}, struct_len={struct_len}, source={structure_data[uid]['source']}")
    print(f"  resseq range: {min(resseq_values)} to {max(resseq_values)}")
    print(f"  resseq gaps (non-consecutive): {sum(1 for i in range(1,len(resseq_values)) if resseq_values[i] != resseq_values[i-1]+1)}")
    print()

In [ ]:
def get_aligned_subsequence(full_seq, residues):
    """
    Given the full UniProt sequence and the structure's residue list
    (with resseq numbers), returns the slice of full_seq that should
    correspond to the structure, using resseq as a 1-indexed position.
    """
    resseq_values = [r['resseq'] for r in residues]
    start = min(resseq_values)
    end = max(resseq_values)

    # UniProt sequences are 1-indexed in resseq terms; Python strings are 0-indexed
    aligned_seq = full_seq[start-1:end]
    return aligned_seq, start, end


# Re-check the same 5 mismatches with proper alignment
for uid, seq_len, struct_len in length_mismatches[:5]:
    full_seq = protein_sequences_esm[uid]
    residues = full_structure_data[uid]['residues']
    aligned_seq, start, end = get_aligned_subsequence(full_seq, residues)
    print(f"{uid}: aligned_seq_len={len(aligned_seq)}, struct_len={struct_len}, match={len(aligned_seq)==struct_len}")

In [ ]:
# P05387: aligned_seq_len=0 -- something is fundamentally wrong with the slice
full_seq = protein_sequences_esm['P05387']
print(f"P05387 full sequence length: {len(full_seq)}")
residues = full_structure_data['P05387']['residues']
resseq_values = [r['resseq'] for r in residues]
print(f"resseq range: {min(resseq_values)} to {max(resseq_values)}")
print(f"That means slice would be full_seq[{min(resseq_values)-1}:{max(resseq_values)}]")

In [ ]:
# P04844 and P98161: aligned_seq_len > struct_len even after windowing --
# suggests gaps WITHIN the resseq range (missing residues mid-structure,
# not just at the ends), so the window is right but not every position
# in that window is actually present
for uid in ['P04844', 'P98161']:
    residues = full_structure_data[uid]['residues']
    resseq_values = [r['resseq'] for r in residues]
    expected_count = max(resseq_values) - min(resseq_values) + 1
    actual_count = len(resseq_values)
    print(f"{uid}: window implies {expected_count} residues, structure has {actual_count} -- gap of {expected_count - actual_count}")

In [ ]:
def align_sequence_to_structure(full_seq, residues):
    """
    For each residue in the structure (by resseq), pulls the corresponding
    character from the full UniProt sequence at position (resseq - 1).
    Returns the aligned per-residue sequence (same length/order as residues),
    or None if the resseq numbering doesn't fit within the sequence at all.
    """
    aligned_chars = []
    for r in residues:
        pos = r['resseq'] - 1  # convert to 0-indexed
        if pos < 0 or pos >= len(full_seq):
            return None  # numbering scheme doesn't fit this sequence -- invalid
        aligned_chars.append(full_seq[pos])
    return ''.join(aligned_chars)


def verify_alignment(uid, aligned_seq, residues):
    """Cross-check: does the aligned sequence's amino acids match the
    residue names stored in the structure? This catches numbering schemes
    that fit length-wise but are still wrong (off-by-some-constant)."""
    AA_3TO1 = {
        'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E',
        'GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F',
        'PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V'
    }
    mismatches = 0
    for i, r in enumerate(residues):
        expected = AA_3TO1.get(r['resname'], 'X')
        if aligned_seq[i] != expected:
            mismatches += 1
    return mismatches


# Test on our known cases
for uid in ['P05387', 'O43143', 'P62805', 'P04844', 'P98161']:
    full_seq = protein_sequences_esm[uid]
    residues = full_structure_data[uid]['residues']
    aligned = align_sequence_to_structure(full_seq, residues)

    if aligned is None:
        print(f"{uid}: INVALID -- resseq numbering doesn't fit sequence at all")
    else:
        mismatches = verify_alignment(uid, aligned, residues)
        print(f"{uid}: aligned_len={len(aligned)}, struct_len={len(residues)}, AA mismatches={mismatches}")

In [ ]:
uid = 'P62805'
full_seq = protein_sequences_esm[uid]
residues = full_structure_data[uid]['residues']
resseq_values = [r['resseq'] for r in residues]

print(f"Full sequence ({len(full_seq)} aa):\n{full_seq}\n")
print(f"resseq range: {min(resseq_values)} to {max(resseq_values)}")

print(f"\nFirst 10 residues in structure (resname, resseq):")
for r in residues[:10]:
    print(f"  {r['resname']} at resseq {r['resseq']}, full_seq[{r['resseq']-1}] = {full_seq[r['resseq']-1]}")

print(f"\nWhat AA does the structure actually have at each position vs sequence?")
AA_3TO1 = {
    'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E',
    'GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F',
    'PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V'
}
struct_seq = ''.join(AA_3TO1.get(r['resname'], 'X') for r in residues)
print(f"Structure sequence: {struct_seq}")
print(f"From full_seq window: {full_seq[min(resseq_values)-1:max(resseq_values)]}")

In [ ]:
# Test shifting by 1 to see if it resolves cleanly
def align_with_offset(full_seq, residues, offset):
    aligned_chars = []
    for r in residues:
        pos = r['resseq'] - 1 + offset
        if pos < 0 or pos >= len(full_seq):
            return None
        aligned_chars.append(full_seq[pos])
    return ''.join(aligned_chars)

for offset in [-2, -1, 0, 1, 2]:
    aligned = align_with_offset(full_seq, residues, offset)
    if aligned:
        mismatches = sum(1 for i, r in enumerate(residues) if aligned[i] != AA_3TO1.get(r['resname'], 'X'))
        print(f"offset={offset:+d}: mismatches={mismatches}/{len(residues)}")
    else:
        print(f"offset={offset:+d}: out of bounds")

In [ ]:
def find_best_alignment(full_seq, residues, max_offset=5, max_mismatch_rate=0.05):
    """
    Searches a small range of offsets to find the one that best aligns
    the structure's residues to the full UniProt sequence, verified by
    actual amino acid identity (not just length).

    Returns (aligned_seq, offset, mismatch_count) on success,
    or (None, None, None) if no offset gives an acceptably low mismatch rate.
    """
    best_offset = None
    best_mismatches = None
    best_aligned = None

    for offset in range(-max_offset, max_offset + 1):
        aligned_chars = []
        valid = True
        for r in residues:
            pos = r['resseq'] - 1 + offset
            if pos < 0 or pos >= len(full_seq):
                valid = False
                break
            aligned_chars.append(full_seq[pos])
        if not valid:
            continue

        aligned = ''.join(aligned_chars)
        mismatches = sum(1 for i, r in enumerate(residues) if aligned[i] != AA_3TO1.get(r['resname'], 'X'))

        if best_mismatches is None or mismatches < best_mismatches:
            best_offset = offset
            best_mismatches = mismatches
            best_aligned = aligned

    if best_aligned is None:
        return None, None, None

    mismatch_rate = best_mismatches / len(residues)
    if mismatch_rate > max_mismatch_rate:
        return None, None, None  # even the best offset is too unreliable

    return best_aligned, best_offset, best_mismatches


# Re-test on all 5 known cases
for uid in ['P05387', 'O43143', 'P62805', 'P04844', 'P98161']:
    full_seq = protein_sequences_esm[uid]
    residues = full_structure_data[uid]['residues']
    aligned, offset, mismatches = find_best_alignment(full_seq, residues)

    if aligned is None:
        print(f"{uid}: FAILED to align (excluded)")
    else:
        print(f"{uid}: success, offset={offset:+d}, mismatches={mismatches}/{len(residues)}")

In [ ]:
import time

aligned_sequences = {}
alignment_failures = []
offset_distribution = {}

start = time.time()
for i, uid in enumerate(needed_proteins_list):
    full_seq = protein_sequences_esm[uid]
    residues = full_structure_data[uid]['residues']

    aligned, offset, mismatches = find_best_alignment(full_seq, residues)

    if aligned is None:
        alignment_failures.append(uid)
    else:
        aligned_sequences[uid] = aligned
        offset_distribution[offset] = offset_distribution.get(offset, 0) + 1

    if (i+1) % 500 == 0:
        print(f"  Processed {i+1}/{len(needed_proteins_list)} ({time.time()-start:.1f}s)")

print(f"\nDone. Successfully aligned: {len(aligned_sequences):,}")
print(f"Failed to align (excluded): {len(alignment_failures):,}")
print(f"\nOffset distribution: {dict(sorted(offset_distribution.items()))}")

In [ ]:
# Check which of the 375 failures are AF2/ESMFold (shouldn't happen --
# those should align perfectly since resseq is always 1-indexed sequential
# for predicted structures) vs PDB (expected, since PDB has the numbering issue)
failure_sources = {}
for uid in alignment_failures:
    src = structure_data[uid]['source']
    failure_sources[src] = failure_sources.get(src, 0) + 1

print(f"Alignment failures by source: {failure_sources}")

In [ ]:
# Check pair-level impact
pairs_affected = pairs_split_final_clean[
    pairs_split_final_clean['protein1'].isin(alignment_failures) |
    pairs_split_final_clean['protein2'].isin(alignment_failures)
]
print(f"Pairs affected by these {len(alignment_failures)} protein exclusions: {len(pairs_affected):,}")
print(f"\nSplit distribution of affected pairs:")
print(pairs_affected['split'].value_counts())
print(f"\nLabel distribution of affected pairs:")
print(pairs_affected['label'].value_counts())

In [ ]:
# Finalize: drop the 375 unalignable proteins and their pairs
pairs_final_aligned = pairs_split_final_clean[
    ~pairs_split_final_clean['protein1'].isin(alignment_failures) &
    ~pairs_split_final_clean['protein2'].isin(alignment_failures)
].copy()

print(f"Pairs before alignment filtering: {len(pairs_split_final_clean):,}")
print(f"Pairs after alignment filtering:  {len(pairs_final_aligned):,}")
print(f"\nFinal split distribution:")
print(pairs_final_aligned['split'].value_counts())
print(f"\nLabel distribution per split:")
print(pairs_final_aligned.groupby('split')['label'].value_counts())

# Update the working protein set to only those with valid aligned sequences
final_proteins = set(pairs_final_aligned['protein1']) | set(pairs_final_aligned['protein2'])
print(f"\nFinal protein count needed: {len(final_proteins):,}")

In [ ]:
import os
base_dir = '/content/drive/MyDrive/ppi_gnn'

pairs_final_aligned.to_csv(f'{base_dir}/data/labels/pairs_final_aligned.tsv', sep='\t', index=False)
pairs_final_aligned.to_parquet(f'{base_dir}/data/labels/pairs_final_aligned.parquet', index=False)

import pickle
with open(f'{base_dir}/data/processed/aligned_sequences.pkl', 'wb') as f:
    pickle.dump(aligned_sequences, f)

print("Saved final pairs and aligned sequences")
print(f"Pairs file size: {os.path.getsize(f'{base_dir}/data/labels/pairs_final_aligned.parquet')/1e6:.2f} MB")

In [ ]:
final_proteins_list = list(final_proteins)
print(f"Generating embeddings for {len(final_proteins_list):,} proteins")

esm_embeddings = {}
esm_errors = []

BATCH_SIZE = 8  # safe for T4 with t12; sequences vary in length so this is conservative

import time
start = time.time()

for batch_start in range(0, len(final_proteins_list), BATCH_SIZE):
    batch_uids = final_proteins_list[batch_start:batch_start + BATCH_SIZE]
    batch_data = [(uid, aligned_sequences[uid]) for uid in batch_uids]

    try:
        labels, strs, tokens = batch_converter(batch_data)
        tokens = tokens.cuda()

        with torch.no_grad():
            results = model(tokens, repr_layers=[12])
        reps = results["representations"][12]  # (batch, max_len+2, 480)

        for i, uid in enumerate(batch_uids):
            seq_len = len(aligned_sequences[uid])
            # strip BOS (position 0) and EOS/padding (after seq_len)
            emb = reps[i, 1:seq_len+1, :].cpu().numpy()
            esm_embeddings[uid] = emb

    except Exception as e:
        for uid in batch_uids:
            esm_errors.append((uid, str(e)))

    if (batch_start // BATCH_SIZE + 1) % 50 == 0:
        elapsed = time.time() - start
        print(f"  Processed {batch_start + len(batch_uids)}/{len(final_proteins_list)} ({elapsed:.1f}s, {len(esm_errors)} errors)")

print(f"\nDone. Successful: {len(esm_embeddings):,}")
print(f"Errors: {len(esm_errors):,}")
if esm_errors:
    print(f"Sample errors: {esm_errors[:3]}")

In [ ]:
print(f"Embedding dim check: {esm_embeddings[final_proteins_list[0]].shape}")
print(f"Expect: (n_residues, 480)")

# Verify per-residue count matches the aligned sequence / structure length
sample_uid = final_proteins_list[0]
print(f"\n{sample_uid}: embedding rows={esm_embeddings[sample_uid].shape[0]}, "
      f"aligned_seq_len={len(aligned_sequences[sample_uid])}, "
      f"struct_residues={len(full_structure_data[sample_uid]['residues'])}")

In [ ]:
import pickle
import os

with open(f'{base_dir}/data/processed/esm_embeddings_step8d.pkl', 'wb') as f:
    pickle.dump(esm_embeddings, f)

print(f"Saved ESM-2 embeddings for {len(esm_embeddings):,} proteins")
print(f"File size: {os.path.getsize(f'{base_dir}/data/processed/esm_embeddings_step8d.pkl') / 1e6:.1f} MB")

In [ ]:
final_node_features = {}
merge_errors_final = []

for uid in final_proteins_list:
    try:
        feat_combined = combined_features[uid]      # (n, 37) -- AA+dihedral+pLDDT+source+SS+SASA
        feat_esm = esm_embeddings[uid]                # (n, 480)

        if feat_combined.shape[0] != feat_esm.shape[0]:
            merge_errors_final.append((uid, f"shape mismatch: {feat_combined.shape[0]} vs {feat_esm.shape[0]}"))
            continue

        final_node_features[uid] = np.concatenate([feat_combined, feat_esm], axis=1)  # (n, 517)
    except KeyError as e:
        merge_errors_final.append((uid, f"missing key: {e}"))

print(f"Final node features built: {len(final_node_features):,}")
print(f"Merge errors: {len(merge_errors_final):,}")
if merge_errors_final:
    print(f"Sample errors: {merge_errors_final[:5]}")

# Confirm final shape
sample_uid = final_proteins_list[0]
print(f"\n{sample_uid} final node feature shape: {final_node_features[sample_uid].shape}  (expect (n, 517))")

In [ ]:
import pickle
import os

with open(f'{base_dir}/data/processed/final_node_features.pkl', 'wb') as f:
    pickle.dump(final_node_features, f)

print(f"Saved final node features for {len(final_node_features):,} proteins")
print(f"File size: {os.path.getsize(f'{base_dir}/data/processed/final_node_features.pkl') / 1e6:.1f} MB")

In [ ]:
from scipy.spatial.distance import cdist

def build_edges(uid, full_structure_data):
    residues = full_structure_data[uid]['residues']
    n = len(residues)
    ca_coords = np.array([r['CA'] for r in residues])

    edges = []  # list of (i, j, edge_type)

    # Type 0 -- backbone (sequential), bidirectional
    for i in range(n - 1):
        edges.append((i, i+1, 0))
        edges.append((i+1, i, 0))

    # Type 1 -- spatial contact (Cα-Cα ≤ 8Å, |i-j| > 5)
    dist_matrix = cdist(ca_coords, ca_coords)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            if abs(i - j) > 5 and dist_matrix[i, j] <= 8.0:
                edges.append((i, j, 1))

    # Type 2 -- disulfide bonds (Cys Sγ-Sγ ≤ 2.1Å)
    cys_indices = [i for i, r in enumerate(residues) if r['resname'] == 'CYS' and r['SG'] is not None]
    for idx_a in range(len(cys_indices)):
        for idx_b in range(idx_a + 1, len(cys_indices)):
            i, j = cys_indices[idx_a], cys_indices[idx_b]
            sg_i, sg_j = np.array(residues[i]['SG']), np.array(residues[j]['SG'])
            dist = np.linalg.norm(sg_i - sg_j)
            if dist <= 2.1:
                edges.append((i, j, 2))
                edges.append((j, i, 2))

    # Type 3 -- hydrogen bonds (backbone N-O proxy, ≤ 3.5Å)
    n_coords = np.array([r['N'] if r['N'] is not None else [np.nan]*3 for r in residues])
    o_coords = np.array([r['O'] if r['O'] is not None else [np.nan]*3 for r in residues])
    no_dist = cdist(n_coords, o_coords)
    for i in range(n):
        for j in range(n):
            if i == j or abs(i - j) <= 1:
                continue
            if not np.isnan(no_dist[i, j]) and no_dist[i, j] <= 3.5:
                edges.append((i, j, 3))

    return edges, ca_coords


# Test on our verified protein
test_uid = 'Q5TE82'
edges, ca_coords = build_edges(test_uid, full_structure_data)

from collections import Counter
edge_type_counts = Counter(e[2] for e in edges)
print(f"Total edges: {len(edges):,}")
print(f"By type: {dict(edge_type_counts)}")
print(f"  Type 0 (backbone):   {edge_type_counts.get(0,0)}")
print(f"  Type 1 (spatial):    {edge_type_counts.get(1,0)}")
print(f"  Type 2 (disulfide):  {edge_type_counts.get(2,0)}")
print(f"  Type 3 (H-bond):     {edge_type_counts.get(3,0)}")
print(f"\nResidues in protein: {ca_coords.shape[0]}")
print(f"Edges per residue (rough density check): {len(edges)/ca_coords.shape[0]:.1f}")

In [ ]:
# Check the actual distance distribution -- is 8Å too strict, or is something
# else going on (e.g., coordinates in the wrong units, or a very elongated structure)
ca_coords = np.array([r['CA'] for r in full_structure_data[test_uid]['residues']])
dist_matrix = cdist(ca_coords, ca_coords)

# Exclude diagonal and immediate neighbors for a fair check
n = len(ca_coords)
mask = np.abs(np.subtract.outer(np.arange(n), np.arange(n))) > 5

distances_beyond_seq5 = dist_matrix[mask]
print(f"Distance stats (residue pairs with |i-j|>5):")
print(f"  Min: {distances_beyond_seq5.min():.1f} Å")
print(f"  Median: {np.median(distances_beyond_seq5):.1f} Å")
print(f"  Mean: {distances_beyond_seq5.mean():.1f} Å")
print(f"  % within 8Å: {100*(distances_beyond_seq5 <= 8.0).mean():.2f}%")
print(f"  % within 12Å: {100*(distances_beyond_seq5 <= 12.0).mean():.2f}%")

In [ ]:
# Cross-check with 2-3 other proteins to see if low density is universal or protein-specific
import random
sample_uids = random.sample(final_proteins_list, 3)

for uid in sample_uids:
    edges, ca_coords = build_edges(uid, full_structure_data)
    n = ca_coords.shape[0]
    edge_type_counts = Counter(e[2] for e in edges)
    print(f"{uid}: n={n}, total_edges={len(edges)}, edges/residue={len(edges)/n:.1f}, "
          f"spatial={edge_type_counts.get(1,0)}, spatial/residue={edge_type_counts.get(1,0)/n:.1f}")

In [ ]:
import time

all_edges = {}
edge_build_errors = []

start = time.time()
for i, uid in enumerate(final_proteins_list):
    try:
        edges, ca_coords = build_edges(uid, full_structure_data)
        all_edges[uid] = edges
    except Exception as e:
        edge_build_errors.append((uid, str(e)))

    if (i+1) % 300 == 0:
        print(f"  Processed {i+1}/{len(final_proteins_list)} ({time.time()-start:.1f}s)")

print(f"\nDone. Successful: {len(all_edges):,}")
print(f"Errors: {len(edge_build_errors):,}")
if edge_build_errors:
    print(f"Sample errors: {edge_build_errors[:5]}")

# Overall edge statistics across the dataset
total_edges_all = sum(len(e) for e in all_edges.values())
total_residues_all = sum(len(full_structure_data[uid]['residues']) for uid in all_edges)
print(f"\nTotal edges across dataset: {total_edges_all:,}")
print(f"Total residues across dataset: {total_residues_all:,}")
print(f"Overall edges/residue ratio: {total_edges_all/total_residues_all:.2f}")

type_totals = Counter()
for edges in all_edges.values():
    type_totals.update(e[2] for e in edges)
print(f"\nEdge type totals across dataset:")
for t, name in [(0,'backbone'), (1,'spatial'), (2,'disulfide'), (3,'H-bond')]:
    print(f"  Type {t} ({name}): {type_totals.get(t,0):,}")

In [ ]:
import pickle
import os

with open(f'{base_dir}/data/processed/all_edges.pkl', 'wb') as f:
    pickle.dump(all_edges, f)

print(f"Saved edges for {len(all_edges):,} proteins")
print(f"File size: {os.path.getsize(f'{base_dir}/data/processed/all_edges.pkl') / 1e6:.1f} MB")

Distance encoding    — RBF, 16-dim (16 Gaussians spanning 0-20Å)
Unit displacement     — 3-dim vector (i → j direction), handled as a GVP vector feature
Sequence separation   — 5-dim one-hot bucket
Edge type             — 4-dim one-hot

In [ ]:
def rbf_encode(distances, n_centers=16, max_dist=20.0, sigma=1.5):
    centers = np.linspace(0, max_dist, n_centers)
    distances = np.array(distances).reshape(-1, 1)
    return np.exp(-((distances - centers) ** 2) / (2 * sigma ** 2))


def seq_sep_bucket(i, j):
    sep = abs(i - j)
    vec = np.zeros(5)
    if sep == 1:
        vec[0] = 1
    elif sep <= 5:
        vec[1] = 1
    elif sep <= 12:
        vec[2] = 1
    elif sep <= 24:
        vec[3] = 1
    else:
        vec[4] = 1
    return vec


def build_edge_features(uid, edges, full_structure_data):
    residues = full_structure_data[uid]['residues']
    ca_coords = np.array([r['CA'] for r in residues])

    edge_index = np.array([[e[0] for e in edges], [e[1] for e in edges]])  # (2, E)
    edge_types = np.array([e[2] for e in edges])

    i_idx = edge_index[0]
    j_idx = edge_index[1]

    pos_i = ca_coords[i_idx]
    pos_j = ca_coords[j_idx]
    displacement = pos_j - pos_i
    distances = np.linalg.norm(displacement, axis=1)

    unit_vectors = displacement / (distances.reshape(-1, 1) + 1e-8)
    rbf_feat = rbf_encode(distances)  # (E, 16)
    seq_sep_feat = np.array([seq_sep_bucket(i, j) for i, j in zip(i_idx, j_idx)])  # (E, 5)
    edge_type_onehot = np.eye(4)[edge_types]  # (E, 4)

    edge_scalar_feat = np.concatenate([rbf_feat, seq_sep_feat, edge_type_onehot], axis=1)  # (E, 25)

    return {
        'edge_index': edge_index,        # (2, E)
        'edge_scalar': edge_scalar_feat, # (E, 25)
        'edge_vector': unit_vectors,      # (E, 3) -- the geometric vector feature for GVP
        'edge_type': edge_types           # (E,)
    }


# Test on our verified protein
test_uid = 'Q5TE82'
edge_feat = build_edge_features(test_uid, all_edges[test_uid], full_structure_data)

print(f"edge_index shape: {edge_feat['edge_index'].shape}")
print(f"edge_scalar shape: {edge_feat['edge_scalar'].shape}  (expect (E, 25))")
print(f"edge_vector shape: {edge_feat['edge_vector'].shape}  (expect (E, 3))")
print(f"\nSample edge scalar features (first edge): {edge_feat['edge_scalar'][0]}")
print(f"Sample edge vector (should be unit length): {np.linalg.norm(edge_feat['edge_vector'][0]):.4f}")

In [ ]:
# Confirm: is the first edge in the list actually a backbone edge or a spatial edge?
first_edge = all_edges[test_uid][0]
print(f"First edge raw tuple (i, j, edge_type): {first_edge}")
print(f"i={first_edge[0]}, j={first_edge[1]}, type={first_edge[2]}")

In [ ]:
import time

all_edge_features = {}
edge_feature_errors = []

start = time.time()
for i, uid in enumerate(final_proteins_list):
    try:
        all_edge_features[uid] = build_edge_features(uid, all_edges[uid], full_structure_data)
    except Exception as e:
        edge_feature_errors.append((uid, str(e)))

    if (i+1) % 300 == 0:
        print(f"  Processed {i+1}/{len(final_proteins_list)} ({time.time()-start:.1f}s)")

print(f"\nDone. Successful: {len(all_edge_features):,}")
print(f"Errors: {len(edge_feature_errors):,}")
if edge_feature_errors:
    print(f"Sample errors: {edge_feature_errors[:5]}")

In [ ]:
import pickle
import os

with open(f'{base_dir}/data/processed/all_edge_features.pkl', 'wb') as f:
    pickle.dump(all_edge_features, f)

print(f"Saved edge features for {len(all_edge_features):,} proteins")
print(f"File size: {os.path.getsize(f'{base_dir}/data/processed/all_edge_features.pkl') / 1e6:.1f} MB")

In [ ]:
!pip install torch_geometric --quiet

In [ ]:
import torch
from torch_geometric.data import Data
import os

os.makedirs(f'{base_dir}/data/processed/graphs', exist_ok=True)

def assemble_graph(uid):
    node_feat = final_node_features[uid]          # (n, 517)
    edge_data = all_edge_features[uid]
    residues = full_structure_data[uid]['residues']
    ca_coords = np.array([r['CA'] for r in residues])

    data = Data(
        x=torch.tensor(node_feat, dtype=torch.float32),
        pos=torch.tensor(ca_coords, dtype=torch.float32),
        edge_index=torch.tensor(edge_data['edge_index'], dtype=torch.long),
        edge_attr=torch.tensor(edge_data['edge_scalar'], dtype=torch.float32),
        edge_vector=torch.tensor(edge_data['edge_vector'], dtype=torch.float32),
        edge_type=torch.tensor(edge_data['edge_type'], dtype=torch.long),
    )
    data.uniprot_id = uid
    data.n_residues = node_feat.shape[0]
    data.source = structure_data[uid]['source']

    return data


# Test on our verified protein
test_uid = 'Q5TE82'
test_graph = assemble_graph(test_uid)
print(test_graph)
print(f"\nx.shape: {test_graph.x.shape}")
print(f"edge_index.shape: {test_graph.edge_index.shape}")
print(f"edge_attr.shape: {test_graph.edge_attr.shape}")
print(f"edge_vector.shape: {test_graph.edge_vector.shape}")

In [ ]:
import time

graph_save_errors = []
saved_count = 0

start = time.time()
for i, uid in enumerate(final_proteins_list):
    try:
        graph = assemble_graph(uid)
        torch.save(graph, f'{base_dir}/data/processed/graphs/{uid}.pt')
        saved_count += 1
    except Exception as e:
        graph_save_errors.append((uid, str(e)))

    if (i+1) % 300 == 0:
        print(f"  Processed {i+1}/{len(final_proteins_list)} ({time.time()-start:.1f}s)")

print(f"\nDone. Saved: {saved_count:,}")
print(f"Errors: {len(graph_save_errors):,}")
if graph_save_errors:
    print(f"Sample errors: {graph_save_errors[:5]}")

In [ ]:
# Check total disk usage of the saved graphs
import subprocess
result = subprocess.run(['du', '-sh', f'{base_dir}/data/processed/graphs/'], capture_output=True, text=True)
print(result.stdout)

In [ ]:
# Pick a real pair from the final dataset
sample_pair = pairs_final_aligned.iloc[0]
uid_a, uid_b, label, split = sample_pair['protein1'], sample_pair['protein2'], sample_pair['label'], sample_pair['split']

graph_a = torch.load(f'{base_dir}/data/processed/graphs/{uid_a}.pt', weights_only=False)
graph_b = torch.load(f'{base_dir}/data/processed/graphs/{uid_b}.pt', weights_only=False)

print(f"Pair: {uid_a} <-> {uid_b}")
print(f"Label: {label} ({'interacting' if label==1 else 'non-interacting'})")
print(f"Split: {split}")
print(f"\nProtein A: {graph_a.x.shape[0]} residues, source={graph_a.source}")
print(f"Protein B: {graph_b.x.shape[0]} residues, source={graph_b.source}")
print(f"\nBoth graphs load correctly: {graph_a.x.shape[1] == graph_b.x.shape[1] == 517}")